In [7]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

In [8]:
df = pd.read_csv('./Cleaned Data/cleaned_data.csv')

In [9]:
df

,Issue Type,Status,Priority,Summary,Original Time Estimated,Total Time Spent
0,Bug,Done,Blocker,"['When', 'integration', 'option', 'setup', 'su...",0,30
1,Bug,Done,Major,"['Getting', 'null', 'error', 'response', 'firs...",0,65
2,Story,Done,Critical,"['I', 'able', 'setup', 'Finance', 'assistant']",0,0
3,Story,Done,Critical,"['When', 'session', 'expires', 'I', 'taken', '...",60,60
4,Bug,Done,Major,"['All', 'connections', 'deleted', 'blank', 'co...",0,105
...,...,...,...,...,...,...
10292,Subtask,Done,Major,"['Add', 'form', 'creating', 'vendor', 'item']",120,60
10293,Subtask,Done,Major,"['Add', 'command', 'receiver', 'endpoint', 'cr...",120,60
10294,Subtask,In Progress,Major,"['Update', 'expense', 'form', 'show', 'list', ...",120,390
10295,Subtask,Done,Major,"['Add', 'form', 'editing', 'invoice']",120,40


In [10]:
text_features = ['Summary']
categorical_features = ['Issue Type', 'Priority']
target = 'Total Time Spent'

In [11]:
X = df[text_features + categorical_features]
y = df[target]

In [12]:
tfidf_vectorizer = TfidfVectorizer()
one_hot_encoder = OneHotEncoder(handle_unknown='ignore')

In [13]:
#combine all features
preprocessor = ColumnTransformer(
    transformers=[
        ('text', tfidf_vectorizer, 'Summary'),  # Pass a string, not list
        ('cat', one_hot_encoder, categorical_features)
    ]
)

In [14]:
# Build and train model
model = make_pipeline(preprocessor, LinearRegression())

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
model.fit(X_train, y_train)

,steps,"[('columntransformer', ...), ('linearregression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('text', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [16]:
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f"Mean Absolute Error: {mae:.2f}")

Mean Absolute Error: 30.89


In [21]:
print("Predictions:", y_pred)
print("Actual:", y_test.values)
print("MAE:", mean_absolute_error(y_test, y_pred))

Predictions: [ 942.96047679  120.01605945  123.56118855 ...   56.66792217  113.02855929
 -155.12690017]
Actual: [943 120 282 ... 120  90 120]
MAE: 30.89116775970396


In [24]:
pred_series = pd.Series(y_pred, index=X_test.index, name='Predicted Time Spent')
orig_est_series = df.loc[X_test.index, 'Original Time Estimated']

results = pd.DataFrame({
    'Summary': df.loc[X_test.index, 'Summary'],
    'Issue Type': df.loc[X_test.index, 'Issue Type'],
    'Priority': df.loc[X_test.index, 'Priority'],
    'Original Estimated Time': orig_est_series,
    'Actual Time Spent': y_test,
    'Predicted Time Spent': pred_series
})

results['Absolute Error'] = abs(results['Actual Time Spent'] - results['Predicted Time Spent'])


In [25]:
results

,Summary,Issue Type,Priority,Original Estimated Time,Actual Time Spent,Predicted Time Spent,Absolute Error
5317,"['Switch', 'using', 'NodeJS', 'instead']",Subtask,Major,0,943,942.960477,0.039523
3688,"['Make', 'payout', 'transactions', 'pulled']",Subtask,Major,60,120,120.016059,0.016059
8771,"['add', 'endpoint', 'upserting', 'categories']",Subtask,Major,60,282,123.561189,158.438811
3437,"['Remove', 'old', 'accounts', 'alert', 'service']",Task,Critical,0,125,125.017704,0.017704
8191,"['Refactor', 'code', 'structure', 'WeOS', 'v2']",Task,Major,120,405,405.007893,0.007893
...,...,...,...,...,...,...,...
411,"['Update', 'IAM', 'idempotent', 'endpoint', 'A...",Task,Blocker,0,150,149.943906,0.056094
613,"['Update', 'refactor', 'igniters', 'use', 'ser...",Subtask,Major,0,180,179.996900,0.003100
6876,"['push', 'updates', 'staging', 'test']",Subtask,Major,60,120,56.667922,63.332078
5561,"['create', 'event', 'handler', 'creating', 'me...",Subtask,Major,60,90,113.028559,23.028559


In [26]:
results.to_csv('./Cleaned Data/prediction_results.csv', index=False)